# 简介

以下是对 **scikit-rf**（也称为 `skrf`）的简要介绍。目标读者是那些已经熟悉 Python 环境，并且对 Python 有一定了解的人。如果您完全不熟悉 Python，请参阅 SciPy 的《入门指南》（[Getting Started](http://www.scipy.org/Getting_Started)）。首先，导入 scikit-rf 模块 `skrf`，并将其命名为 `rf`。

In [ ]:
import skrf as rf

如果出现错误，请参阅[安装](Installation.ipynb)教程。

## 网络

`skrf` 中的核心对象是一个 N 端口微波 [网络][Network] 对象。可以通过多种方式创建 [网络][Network]，其中一种方式是从存储在 Touchstone 文件中的数据创建。[网络]: ../api/network.rst

In [ ]:
ring_slot = rf.Network('data/ring slot.s2p')

如果找不到 `ring_slot.s2p` 文件，可以直接从 `skrf.data` 模块导入。

In [ ]:
from skrf.data import ring_slot  # noqa: F811

如果将网络信息输入到命令行，则会打印出该网络的简短描述。

In [ ]:
ring_slot

微波[网络](../api/network.rst)的基本属性由以下属性提供：* `Network.s`：散射参数矩阵。* `Network.z0`：端口阻抗矩阵。* `Network.frequency`：频率对象。

[Network](../api/network.rst) 对象还有许多其他的属性和方法，这些可以在其文档字符串中找到。如果您使用的是 IPython/Jupyter，那么这些属性和方法可以在命令行中使用“Tab”键进行自动补全。```In [1]: ring_slot.s<TAB>ring_slot.s              ring_slot.s_arcl         ring_slot.s_imring_slot.s11            ring_slot.s_arcl_unwrap  ring_slot.s_mag...```

构建网络的其他方法在[网络教程](Networks.ipynb)中有详细介绍。

### 线性运算

可以通过重载运算符对散射参数执行逐元素数学运算。为了说明这一点，我们加载几个存储在 `skrf.data` 模块中的 `Network` 对象。

In [ ]:
short = rf.data.wr2p2_short
delayshort = rf.data.wr2p2_delayshort

它们的散射参数之间的复数差值通过以下方式计算：

In [ ]:
short - delayshort

这将返回一个新的 [Network](../api/network.rst) 对象。其他算术运算符也被重载。

In [ ]:
short/delayshort

### 级联和去嵌入

也可以通过运算符来实现两个二端口网络的级联和去嵌入。级联操作通过幂运算符 ``**`` 实现。

In [ ]:
short = rf.data.wr2p2_short
line = rf.data.wr2p2_line

delayshort = line ** short

可以通过级联网络的*逆矩阵*来实现去嵌入。可以通过属性 `Network.inv` 访问网络的逆矩阵。

In [ ]:
short = line.inv ** delayshort

有关 [Network](../api/network.rst) 对象提供的功能的更多信息，例如插值、拼接、n 端口连接和 IO 支持，请参阅 [Networks](Networks.ipynb) 教程。

### 查找网络参数的最小值（或最大值）通常，我们希望找到网络参数（如散射参数、阻抗参数等）的最小值（或最大值）以及发生该值的频率。在 `scikit-rf` 中，网络参数存储为 [Numpy](https://numpy.org/) 数组，其形状为 ($N_F$, $N_p$, $N_p$)，其中 $N_F$ 是频率点的数量，$N_p$ 是网络端口的数量。

In [ ]:
print(type(line.s))  # s-parameters are stored as a Numpy array

In [ ]:
print(line.s.shape)  # line is a 2-port Network defined on 201 frequency points

频率点在 `Network` 的 `frequency` 参数中定义：

In [ ]:
print(line.frequency)  # returns a Frequency object

频率值由 `frequency.f` 参数给出，或者简单地写作 `.f`：

In [ ]:
line.f[0:10]  # the 10 first frequency points. Same than line.frequency.f[0:10]

由于它是 NumPy 数组，因此可以使用 `min()`（或 `max()`）方法来查找 $S_{21}$ 参数幅值的最小值（或最大值）：

In [ ]:
import numpy as np

rs = rf.data.ring_slot  # another 2-port example

print(rs.s_mag[:,1,0].min())  # or .max() for maximum. Watch out that Python indexing starts at 0!

可以使用 Numpy 函数 [`argmin`](https://numpy.org/doc/stable/reference/generated/numpy.argmin.html?highlight=argmin#numpy.argmin) 找到 $S_{11}$ 参数的幅度最小的频率：

In [ ]:
f_match = rs.f[np.argmin(rs.s_mag[:,0,0])]  # frequency for min(|S11|)
print(f_match)

## 绘图

**skrf** 包含一个函数，它可以更新你的 [matplotlib rcParams](http://matplotlib.org/users/customizing.html)，从而使生成的图形看起来与这些教程中的图形相似。

In [ ]:
# display plots in notebook
%matplotlib inline
import matplotlib.pyplot as plt

rf.stylely()

[Network](../api/network.rst) 类的各种方法提供了便捷的方式来绘制网络参数的各个分量：* `Network.plot_s_db()`：以对数刻度绘制散射参数的幅值。* `Network.plot_s_deg()`：以度为单位绘制散射参数的相位。* `Network.plot_s_smith()`：在史密斯圆图上绘制复数散射参数。

在幅度、相位和史密斯图上绘制“ring_slot”的所有四个散射参数。

In [ ]:
ring_slot.plot_s_db()

或者绘制 $S_{12}$ 的相位。

In [ ]:
ring_slot.plot_s_deg(m=0,n=1)

In [ ]:
ring_slot.plot_s_smith(lw=2)
plt.title('Big ole Smith Chart')

有关绘图的更多详细信息，请参阅 [绘图](Plotting.ipynb) 教程。

## 网络集

[NetworkSet](../api/networkSet.rst) 对象表示一个无序的网络集合，并提供了计算统计量和显示不确定度范围的方法。[NetworkSet](../api/networkSet.rst) 可以从 [Network](../api/network.rst) 的列表或字典创建。可以使用 `rf.io.read_all()` 快速完成此操作，该函数会加载目录中所有可由 skrf 读取的对象。参数 `contains` 用于仅加载与给定子字符串匹配的文件。

In [ ]:
rf.io.read_all('data/', contains='ro')

可以直接将此字典传递给 [NetworkSet](../api/networkSet.rst) 构造函数。

In [ ]:
from skrf.networkSet import NetworkSet

ro_dict = rf.io.read_all('data/', contains='ro')
ro_ns = NetworkSet(ro_dict, name='ro set') # name is optional
ro_ns

`NetworkSet` 类似于列表。

### 统计特性

可以通过访问 [NetworkSet](../api/networkSet.rst) 的属性来计算统计量。例如，要计算集合的复数平均值，请访问 ``mean_s`` 属性。

In [ ]:
ro_ns.mean_s

无论输出类型如何，返回的结果都会存储在 [Network](../api/network.rst)s 的散射参数中。类似地，要计算数据集的复数标准差，

In [ ]:
ro_ns.std_s

由于这些方法返回一个 [Network](../api/network.rst) 对象，因此结果可以像处理 [Network](../api/network.rst) 对象一样进行保存或绘制。要绘制数据集的标准偏差的幅度，

In [ ]:
ro_ns.std_s.plot_s_mag(label='S11')
plt.ylabel('Standard Deviation')
plt.title('Standard Deviation of RO')

### 绘制不确定度边界

可以通过以下方法绘制任何网络参数的不确定度范围。

In [ ]:
ro_ns.plot_uncertainty_bounds_s_db(label='S11');

请参阅 [networkset](NetworkSet.ipynb) 教程以获取更多信息。

## 虚拟仪器

[skrf.vi](../api/vi/index.rst) 模块提供了用于与某些仪器通信的类。目前，这仅限于 VNA，但未来可能会添加更多支持。请参阅 [虚拟仪器](VirtualInstruments.ipynb) 教程以获取更多信息。

一个使用 `PNA` 类来获取一些散射参数数据并绘制图表的示例。

```from skrf.vi.vna.keysight import PNAinstr = PNA(address="TCPIP0::10.0.0.1::INSTR")freq = Frequency(start=2.3e9, stop=2.6e9, npoints=401, unit='hz')instr.frequency = freqntwk = instr.get_snp_network(ports=(1,2))ntwk.s21.plot_s_db()```

## 校准

校准是通过一个 [Calibration](../api/calibration/index.rst) 类来执行的。在大多数情况下，创建一个 [Calibration](../api/calibration/index.rst) 对象至少需要以下两个信息：*   一个包含测量 [Network](../api/network.rst) 数据的列表*   一个包含理想 [Network](../api/network.rst) 数据的列表每个列表中的 [Network](../api/network.rst) 元素必须相似（例如，相同的端口数、频率信息等），并且必须相互对应，这意味着理想数据列表的第一个元素必须与测量数据列表的第一个元素对应。下面是一个示例脚本，说明如何创建 [Calibration](../api/calibration/index.rst) 对象。

### 单端口校准

```import skrf as rffrom skrf.calibration import OnePortmy_ideals = rf.io.read_all('ideals/')my_measured = rf.io.read_all('measured/')duts = rf.io.read_all('measured/')## 创建一个校准实例cal = OnePort(    ideals = [my_ideals[k] for k in ['short','open','load']],    measured = [my_measured[k] for k in ['short','open','load']],    )caled_duts = [cal.apply_cal(dut) for dut in duts.values()]有关更多详细信息和示例，请参阅[校准](Calibration.ipynb)教程。```

## 传输线介质

可以通过 [Media](../api/media/index.rst) 类的相关方法创建简单的基于传输线的网络。该类表示给定介质的传输线对象。创建后，[Media](../api/media/index.rst) 对象将包含必要的属性，例如“传播常数”和“特性阻抗”，这些属性用于生成微波电路。基本用法如下：

### 共面波导

In [ ]:
from skrf import Frequency
from skrf.media import CPW, Coaxial

freq = Frequency(75, 110, 101, 'GHz')
cpw =  CPW(freq, w = 10e-6, s = 5e-6, ep_r = 10.6)
cpw

In [ ]:
cpw.line(d=90,unit='deg', name='line')

### 同轴电缆

In [ ]:
freq = Frequency(1, 10, 101, 'GHz')
coax = Coaxial(frequency = freq, Dint = 1e-3, Dout = 2e-3)
coax